In [10]:
# rag_v2.py — Upgraded RAG with:
#   Step 2: Load from PDF / TXT files
#   Step 3: Chunking with overlap
#   Step 4: Persist embeddings to disk
#   Step 5: ChromaDB as vector store

import os
import hashlib
import requests
from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings


In [11]:

# ─── CONFIG ────────────────────────────────────────────────────────────────
OLLAMA_URL   = "http://localhost:11434/api/generate"
MODEL        = "llama3.2"
EMBED_MODEL  = "all-MiniLM-L6-v2"
DOCS_DIR     = "./docs"           # put your PDFs / TXT files here
CHROMA_DIR   = "./chroma_store"   # embeddings persisted here
COLLECTION   = "rag_knowledge"
CHUNK_SIZE   = 500                # characters per chunk
CHUNK_OVERLAP= 50                 # overlap between chunks
TOP_K        = 3                  # chunks to retrieve per query


In [12]:

# ─── STEP 2: LOADERS ───────────────────────────────────────────────────────
def load_txt(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def load_pdf(path: str) -> str:
    from pypdf import PdfReader
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def load_documents(docs_dir: str) -> list[dict]:
    """Load all .txt and .pdf files from a directory."""
    docs = []
    for filepath in Path(docs_dir).glob("**/*"):
        if filepath.suffix.lower() == ".txt":
            text = load_txt(str(filepath))
        elif filepath.suffix.lower() == ".pdf":
            text = load_pdf(str(filepath))
        else:
            continue
        if text.strip():
            docs.append({"source": str(filepath), "text": text})
            print(f"  Loaded: {filepath.name} ({len(text)} chars)")
    return docs

# ─── STEP 3: CHUNKING WITH OVERLAP ─────────────────────────────────────────
def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    """
    Split text into overlapping chunks.
    Overlap ensures context is not lost at chunk boundaries.
    
    Example with chunk_size=10, overlap=3:
      "ABCDEFGHIJKLMNO"
       chunk1: ABCDEFGHIJ
       chunk2: HIJKLMNO      ← starts 3 chars before chunk1 ended
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap   # slide forward, keeping overlap
    return [c for c in chunks if len(c) > 30]  # drop tiny trailing chunks

def chunk_documents(docs: list[dict]) -> list[dict]:
    """Chunk all documents, keeping track of source filename."""
    all_chunks = []
    for doc in docs:
        chunks = chunk_text(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP)
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "id": hashlib.md5(f"{doc['source']}_{i}".encode()).hexdigest(),
                "text": chunk,
                "source": doc["source"],
                "chunk_index": i
            })
    print(f"  Total chunks: {len(all_chunks)}")
    return all_chunks

# ─── STEP 4 + 5: CHROMADB WITH PERSISTENCE ─────────────────────────────────
def get_or_create_collection(chroma_dir: str):
    """
    ChromaDB persists embeddings to disk automatically.
    On first run: embeds everything and saves.
    On subsequent runs: loads from disk instantly — no re-embedding.
    """
    client = chromadb.PersistentClient(path=chroma_dir)
    collection = client.get_or_create_collection(
        name=COLLECTION,
        metadata={"hnsw:space": "cosine"}  # use cosine similarity
    )
    return client, collection

def index_chunks(collection, chunks: list[dict], embedder):
    """Add only NEW chunks to the collection (skip already-indexed ones)."""
    existing_ids = set(collection.get()["ids"])
    new_chunks = [c for c in chunks if c["id"] not in existing_ids]

    if not new_chunks:
        print("  All chunks already indexed. Skipping embedding.")
        return

    print(f"  Embedding {len(new_chunks)} new chunks...")
    texts = [c["text"] for c in new_chunks]
    embeddings = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True)

    collection.add(
        ids=[c["id"] for c in new_chunks],
        embeddings=embeddings.tolist(),
        documents=texts,
        metadatas=[{"source": c["source"], "chunk_index": c["chunk_index"]}
                   for c in new_chunks]
    )
    print(f"  Indexed {len(new_chunks)} chunks into ChromaDB.")

# ─── RETRIEVER ──────────────────────────────────────────────────────────────
def retrieve(query: str, collection, embedder, top_k: int = TOP_K) -> list[dict]:
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    chunks = []
    for text, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunks.append({
            "text": text,
            "source": Path(meta["source"]).name,
            "score": round(1 - dist, 3)   # convert distance → similarity
        })
    return chunks

# ─── GENERATOR ──────────────────────────────────────────────────────────────
def ask(question: str, collection, embedder) -> str:
    retrieved = retrieve(question, collection, embedder)

    # Show what was retrieved (useful for debugging)
    print("\n  [Retrieved chunks]")
    for i, r in enumerate(retrieved):
        print(f"    {i+1}. [{r['source']}] score={r['score']} — {r['text'][:80]}...")

    # Build tight prompt — important for smaller models
    context = "\n\n".join(
        f"[Source: {r['source']}]\n{r['text']}" for r in retrieved
    )
    prompt = f"""Use only the context below to answer the question. Be concise and factual.

Context:
{context}

Question: {question}
Answer:"""

    response = requests.post(OLLAMA_URL, json={
        "model": MODEL,
        "prompt": prompt,
        "stream": False
    })
    return response.json()["response"].strip()

# ─── MAIN ───────────────────────────────────────────────────────────────────
def main():
    print("\n=== RAG v2 — Starting up ===\n")

    # 1. Load embedding model
    print("[1/4] Loading embedding model...")
    embedder = SentenceTransformer(EMBED_MODEL)

    # 2. Load documents from ./docs
    print(f"\n[2/4] Loading documents from '{DOCS_DIR}'...")
    os.makedirs(DOCS_DIR, exist_ok=True)
    docs = load_documents(DOCS_DIR)
    if not docs:
        print(f"  No documents found in '{DOCS_DIR}'.")
        print("  Add some .txt or .pdf files and rerun.")
        return

    # 3. Chunk documents
    print("\n[3/4] Chunking documents...")
    chunks = chunk_documents(docs)

    # 4. Index into ChromaDB (persists to disk)
    print(f"\n[4/4] Indexing into ChromaDB at '{CHROMA_DIR}'...")
    _, collection = get_or_create_collection(CHROMA_DIR)
    index_chunks(collection, chunks, embedder)

    # 5. Query loop
    print("\n=== Ready! Type your question (or 'quit' to exit) ===\n")
    while True:
        question = input("You: ").strip()
        if question.lower() in ("quit", "exit", "q"):
            break
        if not question:
            continue
        answer = ask(question, collection, embedder)
        print(f"\nRAG: {answer}\n")

if __name__ == "__main__":
    main()


=== RAG v2 — Starting up ===

[1/4] Loading embedding model...


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 7702.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[2/4] Loading documents from './docs'...
  Loaded: rag-paper.pdf (69074 chars)

[3/4] Chunking documents...
  Total chunks: 154

[4/4] Indexing into ChromaDB at './chroma_store'...
  Embedding 154 new chunks...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  3.97it/s]


  Indexed 154 chunks into ChromaDB.

=== Ready! Type your question (or 'quit' to exit) ===



You:  what is a RAG ?



  [Retrieved chunks]
    1. [rag-paper.pdf] score=0.6 — edgments
The authors would like to thank the reviewers for their thoughtful and ...
    2. [rag-paper.pdf] score=0.464 — them, showing promise in being applied to a
wide variety of NLP tasks.
9
Broader...
    3. [rag-paper.pdf] score=0.446 — d
BART were factual in a further 17% of cases, clearly demonstrating the effecti...

RAG: A RAG (Recurrent Autoencoder-based Graph) is not explicitly defined in the provided context. However, based on the title of the paper "RAG: Reasoning Autoencoder for Generative Models" and the mention of the "RAG-Token" in the context of Jeopardy questions, it can be inferred that a RAG is likely a model or a component of a model designed to improve generative capabilities in natural language processing tasks.



You:  what else can you say about RAG ? 



  [Retrieved chunks]
    1. [rag-paper.pdf] score=0.628 — edgments
The authors would like to thank the reviewers for their thoughtful and ...
    2. [rag-paper.pdf] score=0.505 — them, showing promise in being applied to a
wide variety of NLP tasks.
9
Broader...
    3. [rag-paper.pdf] score=0.483 — d
BART were factual in a further 17% of cases, clearly demonstrating the effecti...

RAG: Based on the provided context, here are some concise and factual points about RAG:

* RAG (Real-Advantage-Generative) is a generation model that shows promise in applying to a wide variety of NLP tasks.
* It offers several positive societal benefits, including being more grounded in real factual knowledge, offering more control and interpretability, and being able to be employed in various scenarios with direct benefit to society.
* RAG performs better than state-of-the-art generation models, such as BART, with a 17% increase in factual accuracy.
* RAG can generate more specific responses by a large ma

You:  quit
